In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('netflix_catalogue.csv')

print(f"Loaded: {len(df)} titles")
print(df.head())

Loaded: 3000 titles
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


Task 1

In [6]:
# Task 1 — Heatmap: content by rating and release decade

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Keep only the required ratings
ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
heatmap_df = df[df['rating'].isin(ratings_filter)]

# Group and count titles
heatmap_counts = (
    heatmap_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

# Pivot for heatmap
heatmap_pivot = heatmap_counts.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

# Optional: sort decades chronologically
heatmap_pivot = heatmap_pivot.reindex(
    sorted(heatmap_pivot.columns),
    axis=1
)

# Create heatmap
fig = px.imshow(
    heatmap_pivot,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    labels=dict(
        x='Release Decade',
        y='Content Rating',
        color='Number of Titles'
    ),
    title='TV-MA titles dominate Netflix’s modern catalogue while PG films fade after the 2000s'
)

# Annotate one important cell directly
fig.add_annotation(
    x='2010s',
    y='TV-MA',
    text='Largest concentration',
    showarrow=True,
    arrowhead=2,
    font=dict(size=11)
)

fig.update_layout(
    title_font_size=18,
    xaxis_title='Release Decade',
    yaxis_title='Content Rating'
)

fig.show()

Task 2

In [9]:
print(df.columns)

Index(['type', 'release_year', 'added_year', 'genre', 'country', 'rating',
       'duration', 'decade'],
      dtype='object')


In [10]:
# Task 2 — Waterfall: Movie library additions by year

# Filter to Movies only
movies_df = df[
    (df['type'] == 'Movie') &
    (df['added_year'].between(2015, 2022))
]

# Count movies added per year
movie_growth = (
    movies_df
    .groupby('added_year')
    .size()
    .reset_index(name='count')
    .sort_values('added_year')
)

# Find year with largest addition
max_row = movie_growth.loc[movie_growth['count'].idxmax()]

# Create waterfall chart
fig = go.Figure(go.Waterfall(
    name='Movies Added',
    orientation='v',

    measure=['relative'] * len(movie_growth) + ['total'],

    x=list(movie_growth['added_year'].astype(str)) + ['Total'],
    y=list(movie_growth['count']) + [0],

    increasing=dict(marker=dict(color='green')),
    decreasing=dict(marker=dict(color='red')),
    totals=dict(marker=dict(color='blue')),

    text=list(movie_growth['count']) + [movie_growth['count'].sum()],
    textposition='outside'
))

# Annotation
fig.add_annotation(
    x=str(max_row['added_year']),
    y=max_row['count'],
    text=f"Peak growth: {max_row['count']} movies",
    showarrow=True,
    arrowhead=2,
    yshift=15
)

fig.update_layout(
    title='Netflix movie additions surged rapidly after 2016 before slowing by 2022',
    xaxis_title='Year Added',
    yaxis_title='Movies Added',
    title_font_size=18
)

fig.show()